<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/PAL_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> PAL (Linear Models) </b> </font>

---

Using the following data set, determine the <b>best-fit</b> value and <b>RMSE</b> for the <b>Conductance</b> of the circuit from which the data came from.

<br>

The relationship between voltage ($V$) and Conductance ($G$) is given by: $I = G \cdot V$

><b>Recall:</b> Conductance ($G$) is a measure of how easily electric current can flow through a material. It is the opposite of resistance. $G = \frac{1}{R}$

<a name="Define-Useful-Functions"></a>

---

<font size = 6> <b> Define Useful Functions </b> </font>

---

In [ ]:
#@title This cell defines the display_dataframes, plot_data, and root_mean_squared_error functions.

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  item_list  - List of DataFrames to be displayed                                  ##
##            title_list - List of titles for the displayed DataFrames                         ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(item_list, title_list):

  # Create a String for housing the commands to be sent to the display_html() function:

  html_str = ''

  # Loop over the elements in the item_list and title_list:

  for df, title in zip(item_list, title_list):

      # Wrap title and dataframe in a styled html <div>:

      html_str += f'''
      <div style="display: inline-block; margin-right: 20px; vertical-align: top;">
          <h3 style="text-align: center;">{title}</h3>
          {pd.DataFrame(df).head().to_html()}
      </div>
      '''

  # Send the html String to the display_html() function for interpretation:

  display_html(html_str, raw = True)

##=============================================================================================##
## Function:  plot_data                                                                        ##
##                                                                                             ##
## Purpose:   Create a scatterplot with optional model overlays and error bands                ##
##                                                                                             ##
## Input(s):  data          - List of data points [x_data, y_data]                             ##
##            title         - Graph title                                                      ##
##            axis_labels   - Override axis labels [x_label, y_label] (optional)               ##
##            model_list    - List of model predictions to overlay (optional)                  ##
##            color_list    - Colors for each model line (optional)                            ##
##            label_list    - Labels for each model line (optional)                            ##
##            error_display - Show +/- error band around first model (default is False)        ##
##            error         - Error value for shaded band                                      ##
##                                                                                             ##
## Output(s): graph       - Matplotlib axes object, can be used for overplotting               ##
##=============================================================================================##

def plot_data(data, title, axis_labels = [], model_list = [], color_list = [], label_list = [],
              error_display = False, error = 0):

  # Create the Matplotlib figure:

  figure = plt.figure(figsize = (12, 9))

  # Add a graph to the figure:

  graph = figure.add_subplot()

  # Set the graph background Color:

  graph.set_facecolor('lightcyan')

  # Create a scatterplot of the data:

  sns.scatterplot(x = data[0], y = data[1], ax = graph)

  # Overlay model predictions:

  for i in range(0, len(model_list)):

    graph.plot(data[0], model_list[i], color = color_list[i], label = label_list[i])

  # Set the graph title:

  graph.set_title(title, fontsize = 20)

  # Set the x_label and y_label:

  if (axis_labels != []):

    graph.set_xlabel(axis_labels[0], fontsize = 14)

    graph.set_ylabel(axis_labels[1], fontsize = 14)

  # Apply a grid to the graph:

  graph.grid(which = 'both')

  # Adjust the x-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'x', tight = False)

  # Adjust the y-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'y', tight = False)

  # If requested, show the +/- error bounds:

  if ((error_display == True) and model_list != []):

    model_plus_error  = model_list[0] + error
    model_minus_error = model_list[0] - error

    error_df = pd.DataFrame({
        'X-Data': data[0],
        'Model+Error': model_plus_error,
        'Model-Error': model_minus_error
    }).sort_values(by="X-Data")

    graph.fill_between(error_df['X-Data'], error_df['Model+Error'], error_df['Model-Error'],
                       alpha = 0.5, color = (0.6, 0.6, 0.6), label = "Error Bounds")

  # Add the graph legend

  if (label_list != []): graph.legend()

  # Return the graph object

  return graph

##=============================================================================================##
## Function:  root_mean_squared_error                                                          ##
##                                                                                             ##
## Purpose:   Implement the Root Mean Squared Error Loss Function                              ##
##                                                                                             ##
## Input(s):  X           - feature (independent) data                                         ##
##            y           - target (dependent) data                                            ##
##            coefficient - slope of the linear model                                          ##
##            bias        - bias of the linear model                                           ##
##                                                                                             ##
## Output(s): rmse        - root mean squared error                                            ##
##=============================================================================================##

def root_mean_squared_error(X, y, coefficient, bias):

  # Create the model's predictions based on a linear combination of the feature data

  predictions = X * coefficient + bias

  # Calculate the squared error:

  squared_error = (y - predictions)**2

  # Find the mean of the squared error:

  mse = squared_error.mean()

  # Take the square root of the mean squared error:

  rmse = np.sqrt(mse)

  # Return the root mean squared error:

  return rmse

<a name="Data"></a>

---

<font size = 6> <b> The Data </b> </font>

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display_html

##=============================================================================================##
## Load The Full Data Set:                                                                     ##
##=============================================================================================##

github = "https://raw.githubusercontent.com/dr-bankert-augustana/PHYS_200/refs/heads/main/"

url = github + "Data/Module_2/voltage_data_1.csv"

full_data = pd.read_csv(url)

##=============================================================================================##
## Clean the Data and Separate Features and Targets:                                           ##
##=============================================================================================##

# Remove any rows missing data:

cleaned_data = full_data.dropna()

# Identify the feature data:

X = cleaned_data["Voltage (V)"]

# Identify the target data:

y = cleaned_data["Current (A)"]

##=============================================================================================##
## Display the Full Dataset, Feature Data, and Target Data:                                    ##
##=============================================================================================##

# Display the feature and target data:

display_dataframes([full_data, X, y], ["Full Dataset", "Feature Data", "Target Data"])